In [1]:
## 1. DAGSHUB & MLFLOW INITIALIZATION
!pip install dagshub mlflow lightgbm -q

import pandas as pd
import numpy as np
import lightgbm as lgb
import mlflow
import dagshub
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

REPO_OWNER = "ejoba22"  
REPO_NAME = "walmart-sales-forecasting"

dagshub.init(repo_owner=REPO_OWNER, repo_name=REPO_NAME, mlflow=True)

mlflow.set_tracking_uri(f"https://dagshub.com/{REPO_OWNER}/{REPO_NAME}.mlflow")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 89.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 86.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=dec50de8-9967-45bc-8746-50dc71bc7d42&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=d1edf8e4d91127677f6209cd32d491910be7b81e9ac5dae6e055d04872494106




Accessing as tvada22

Initialized MLflow to track repo "ejoba22/walmart-sales-forecasting"

Repository ejoba22/walmart-sales-forecasting initialized!

In [2]:
## Data Load and Split
DATA_BASE_PATH = '/kaggle/input/notebooks/elenejobava/walmart-eda-feature-engineering/'

train_fe = pd.read_parquet(DATA_BASE_PATH + 'train_features.parquet')
test_fe  = pd.read_parquet(DATA_BASE_PATH + 'test_features.parquet')

# Kaggle evaluation metric is Weighted Mean Absolute Error (WMAE)
# Holiday weeks have a weight of 5, normal weeks have a weight of 1
train_fe['weight'] = np.where(train_fe['IsHoliday'] == 1, 5, 1)

# Drop non-feature columns
features = [c for c in train_fe.columns if c not in ['Weekly_Sales', 'Date', 'Store_Dept', 'Type', 'weight']]
target = 'Weekly_Sales'

# Use the last 3 months of train data as our validation set.
split_date = train_fe['Date'].max() - pd.Timedelta(weeks=12)

X_train = train_fe[train_fe['Date'] <= split_date][features]
y_train = train_fe[train_fe['Date'] <= split_date][target]
w_train = train_fe[train_fe['Date'] <= split_date]['weight']

X_val = train_fe[train_fe['Date'] > split_date][features]
y_val = train_fe[train_fe['Date'] > split_date][target]
w_val = train_fe[train_fe['Date'] > split_date]['weight']

print(f"Training shape: {X_train.shape}, Validation shape: {X_val.shape}")

# Create LightGBM Dataset objects (highly memory efficient)
train_data = lgb.Dataset(X_train, label=y_train, weight=w_train)
val_data = lgb.Dataset(X_val, label=y_val, weight=w_val, reference=train_data)

Training shape: (386007, 78), Validation shape: (35563, 78)


In [3]:
## MODEL TRAINING & TRACKING
# Autologging handles saving the model, parameters, and feature importance automatically
mlflow.lightgbm.autolog()

# Set up the parent experiment per your professor's instructions
mlflow.set_experiment("LightGBM_Training")

# Define hyperparameter dictionary
params = {
    'objective': 'regression',
    'metric': 'mae',          # Mean Absolute Error
    'boosting_type': 'gbdt',  # Gradient Boosting Decision Tree
    'learning_rate': 0.05,    # Step size 
    'num_leaves': 63,         # Max leaves per tree (control complexity)
    'feature_fraction': 0.8,  # Randomly sample 80% of features per tree (prevents overfitting)
    'verbose': -1,            # Suppress internal LightGBM warnings
    'random_state': 42
}

print("Starting MLflow run...")
with mlflow.start_run(run_name="LightGBM_Baseline"):
    
    # Train the model
    model = lgb.train(
        params,
        train_data,
        num_boost_round=1000,
        valid_sets=[train_data, val_data],
        valid_names=['train', 'validation'],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=50)
        ]
    )
    
    # Generate validation predictions for final manual metric check
    val_preds = model.predict(X_val)
    
    # Calculate Custom Kaggle WMAE
    wmae = np.sum(w_val * np.abs(y_val - val_preds)) / np.sum(w_val)
    
    # Log the custom final metric to MLflow
    mlflow.log_metric("validation_WMAE", wmae)
    
    print(f"\n✓ Run Complete. Validation WMAE: {wmae:.2f}")

Starting MLflow run...
Training until validation scores don't improve for 50 rounds
[50]	train's l1: 1709.17	validation's l1: 1735.78
[100]	train's l1: 1034.38	validation's l1: 1039.5
[150]	train's l1: 955.619	validation's l1: 984.568
[200]	train's l1: 907.55	validation's l1: 956.625
[250]	train's l1: 877.453	validation's l1: 945.981
[300]	train's l1: 852.453	validation's l1: 937.933
[350]	train's l1: 828.575	validation's l1: 928.045
[400]	train's l1: 807.785	validation's l1: 921.66
[450]	train's l1: 789.626	validation's l1: 916.477
[500]	train's l1: 773.863	validation's l1: 911.737
[550]	train's l1: 759.334	validation's l1: 908.641
[600]	train's l1: 746.347	validation's l1: 905.549
[650]	train's l1: 733.875	validation's l1: 902.171
[700]	train's l1: 722.745	validation's l1: 899.319
[750]	train's l1: 712.344	validation's l1: 896.894
[800]	train's l1: 702.969	validation's l1: 895.35
[850]	train's l1: 693.774	validation's l1: 894.897
[900]	train's l1: 684.986	validation's l1: 892.904
[95

2026/07/11 19:04:33 WARNING mlflow.lightgbm: Failed to infer model signature: could not sample data to infer model signature: please ensure that autologging is enabled before constructing the dataset.
2026/07/11 19:04:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



✓ Run Complete. Validation WMAE: 890.16
🏃 View run LightGBM_Baseline at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/1/runs/9371ba6b054e4dd1825abaafb4eaf994
🧪 View experiment at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/1


In [3]:
## RUN 4: OPTIMIZED LIGHTGBM & WAPE METRIC
mlflow.set_experiment("LightGBM_Training")

def calculate_wape(y_true, y_pred):
    """Calculates Weighted Absolute Percentage Error (Retail Standard)"""
    return (np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))) * 100

# Relaxed parameters: Giving the model enough capacity to learn the massive dataset
params_optimized = {
    'objective': 'regression',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 127,             # [FIX] Increased to allow complex pattern learning
    'min_data_in_leaf': 20,        # [FIX] Relaxed restriction
    'lambda_l1': 0.01,             # [FIX] Lighter penalty
    'lambda_l2': 0.01,             # [FIX] Lighter penalty
    'feature_fraction': 0.8,
    'feature_pre_filter': False,   # [THE FIX] Forces LightGBM to ignore cached dataset filters
    'verbose': -1,
    'random_state': 42
}

print("Starting Optimized MLflow run...")

# Force MLflow to re-attach the autologger for this specific cell
mlflow.lightgbm.autolog()

with mlflow.start_run(run_name="LightGBM_Optimized"):
    
    # Train the restricted model
    model_opt = lgb.train(
        params_optimized,
        train_data,
        num_boost_round=1000,
        valid_sets=[train_data, val_data],
        valid_names=['train', 'validation'],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=50)
        ]
    )
    
    # Generate predictions
    val_preds_opt = model_opt.predict(X_val)
    
    # Calculate Metrics
    wmae_opt = np.sum(w_val * np.abs(y_val - val_preds_opt)) / np.sum(w_val)
    wape_opt = calculate_wape(y_val, val_preds_opt)
    
    # Log custom metrics
    mlflow.log_metric("validation_WMAE", wmae_opt)
    mlflow.log_metric("validation_WAPE_percent", wape_opt)
    
    print(f"Validation WMAE: {wmae_opt:.2f}")
    print(f"Human-Readable Error (WAPE): {wape_opt:.2f}%")

Starting Optimized MLflow run...
Training until validation scores don't improve for 50 rounds
[50]	train's l1: 1662.27	validation's l1: 1688.42
[100]	train's l1: 956.436	validation's l1: 994.217
[150]	train's l1: 868.502	validation's l1: 944.987
[200]	train's l1: 815.293	validation's l1: 927.61
[250]	train's l1: 780.564	validation's l1: 918.15
[300]	train's l1: 749.379	validation's l1: 908.742
[350]	train's l1: 724.672	validation's l1: 904.365
[400]	train's l1: 703.233	validation's l1: 900.411
[450]	train's l1: 684.039	validation's l1: 897.561
[500]	train's l1: 666.62	validation's l1: 895.357
[550]	train's l1: 651.102	validation's l1: 893.409
[600]	train's l1: 637.113	validation's l1: 891.501
[650]	train's l1: 624.806	validation's l1: 891.241
[700]	train's l1: 613.037	validation's l1: 889.842
[750]	train's l1: 601.546	validation's l1: 887.905
[800]	train's l1: 591.412	validation's l1: 887.787
Early stopping, best iteration is:
[764]	train's l1: 598.567	validation's l1: 887.31


2026/07/04 09:04:05 WARNING mlflow.lightgbm: Failed to infer model signature: could not sample data to infer model signature: please ensure that autologging is enabled before constructing the dataset.
2026/07/04 09:04:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Validation WMAE: 887.31
Human-Readable Error (WAPE): 5.67%
🏃 View run LightGBM_Optimized at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/0/runs/5a85886b9a0f41a69d85369ff0aba13c
🧪 View experiment at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/0
